In [0]:
from pyspark.sql import functions as F

In [0]:
BRONZE = "abfss://bronze@pharmadatalakeuk.dfs.core.windows.net"
SILVER = "abfss://silver@pharmadatalakeuk.dfs.core.windows.net"
lims = (spark.read
       .option("header", "true")
       .csv(f"{BRONZE}/lims/lims_results.csv"))


In [0]:
lims_dedup = lims.dropDuplicates()
print(f"Rows: {lims.count()}")
print(f"After :{lims_dedup.count()}")
#print(lims_dedup.select("batch_id").distinct().count())

In [0]:
#display(lims)

In [0]:
lims_trimmed =lims_dedup.select([F.trim(c).alias(c) for c in lims_dedup.columns]) 
#display(lims_trimmed)

In [0]:
# display(
#     lims_trimmed.groupBy("result_status").agg(F.count("result_status"))
# )

In [0]:
result_status_clean = (
                        F.when(F.upper(F.col("result_status")).isin("PASS","P","PASSED"),"Pass")
                        .when(F.upper(F.col("result_status")).isin("FAIL","F","FAILED"),"Fail")
                        .when(F.upper(F.col("result_status"))=="OOS","OOS")
                        .otherwise(F.lit(None))
)

In [0]:
lims_standardised = lims_trimmed.withColumn("batch_id",F.upper(F.col("batch_id")))\
                                .withColumn("result_status",result_status_clean)\
                                .withColumn("release_status",F.initcap(F.lower(F.col("release_status"))))



In [0]:
# display(
#     lims_standardised.groupBy("release_status").agg(F.count("*"))
# )

In [0]:
# display(
#     lims_standardised.groupBy("result_status").agg(F.count("*"))
# )

In [0]:
# display(lims_standardised.withColumn
# ("testt",F.regexp_extract(F.col("test_result"), r"(-?\d+(?:\.\d+)?)", 1)))

In [0]:
lims_typed = (lims_standardised.withColumn("below_reporting_limit",F.col("test_result").startswith("<"))
            .withColumn("test_result",F.regexp_extract("test_result",r"(-?\d+(?:\.\d+)?)" , 1).try_cast("double"))
            .withColumn("lower_spec_limit",F.col("lower_spec_limit").try_cast("double"))
            .withColumn("upper_spec_limit",F.col("upper_spec_limit").try_cast("double"))
            .withColumn("limits_swapped",F.when(F.col("lower_spec_limit")>F.col("upper_spec_limit"),True)
                        .otherwise(False))
            .withColumns({
                            "lower_spec_limit": F.when(F.col("limits_swapped"), F.col("upper_spec_limit"))
                                                .otherwise(F.col("lower_spec_limit")),
                            "upper_spec_limit": F.when(F.col("limits_swapped"), F.col("lower_spec_limit"))
                                                .otherwise(F.col("upper_spec_limit"))
                        })
            
            )

In [0]:
print("Below reporting limit:", lims_typed.filter(F.col("below_reporting_limit")).count())
print("Null test_result:     ", lims_typed.filter(F.col("test_result").isNull()).count())
print("Limits swapped:       ", lims_typed.filter(F.col("limits_swapped")).count())
print("Still reversed:       ", lims_typed.filter(F.col("lower_spec_limit") > F.col("upper_spec_limit")).count())
print("One limit missing:    ", lims_typed.filter(F.col("lower_spec_limit").isNull() | F.col("upper_spec_limit").isNull()).count())

In [0]:
# display(lims_typed.select("test_date").distinct().limit(20))

In [0]:
def parse_date(column_name):
    col = F.col(column_name)
    return F.coalesce(
            F.try_to_timestamp(col,F.lit("yyyy-MM-dd")),
            F.try_to_timestamp(col,F.lit("dd/MM/yyyy")),
            F.try_to_timestamp(col,F.lit("dd-MMM-yyyy"))       
    ).cast("date")

In [0]:
lims_parsed = lims_typed.withColumn("test_date",parse_date("test_date"))
# display(lims_parsed)

In [0]:
lims_parsed.printSchema()

In [0]:
lims_final =(lims_parsed
    .withColumn("calculated_status",
                F.when((F.col("test_result").isNull())|(F.col("lower_spec_limit").isNull())|(F.col("upper_spec_limit").isNull()),F.lit(None))
                .when(((F.col("test_result"))>=F.col("lower_spec_limit")) &(F.col("test_result")<=(F.col("upper_spec_limit"))),"Pass")
                .otherwise("Fail") )       
    .withColumn("status_mismatch", F.when(((F.col("result_status")=="Pass") & (F.col("calculated_status")=="Fail"))|((F.col("result_status").isin("Fail","OOS"))&(F.col("calculated_status")=="Pass")),True)
    .otherwise(False))                         
)                         

In [0]:
# display(lims_final.groupBy("status_mismatch").count())

In [0]:
mes_batches = (spark.read.format("delta").load(f"{SILVER}/mes")
               .select("batch_id","start_time")
               .withColumnRenamed("start_time","batch_start_time")
               .withColumn("in_mes",F.lit(True))
               )
               

In [0]:
lims_silver= (lims_final.join(mes_batches, on="batch_id",how="left")
                .withColumn("is_orphan", F.col("in_mes").isNull())
                .drop("in_mes")
                .withColumn("test_before_batch_start",
                            F.when((F.col("test_date")) < (F.to_date(F.col("batch_start_time"))),True)
                            .otherwise(False))
                .drop("batch_start_time")
)

In [0]:
lims_silver.write.format("delta").mode("overwrite").save(f"{SILVER}/lims")

In [0]:
spark.read.format("delta").load(f"{SILVER}/lims").printSchema()

In [0]:
(lims_silver.write
 .mode("overwrite")
 .parquet(f"{SILVER}/export/lims"))
 
 
# display(lims_silver)
